In [ ]:
from ase import Atoms 
from ase.io import read 
from openbabel import openbabel as ob 
from monty.serialization import loadfn 
from collections import Counter,defaultdict
from tqdm.auto import tqdm 
import pandas as pd 
import glob 

### 1. Take an individual ase.atoms.Atoms object and determine if there are multiple molecules present

In [4]:
def seperate_atoms(atoms: Atoms) -> list[Atoms]:
    """Split an ase.Atoms object into its disconnected molecular fragments."""
    mol = ob.OBMol()
    for Z, pos in zip(atoms.numbers, atoms.positions):
        a = mol.NewAtom()
        a.SetAtomicNum(int(Z))
        a.SetVector(*map(float, pos))
    mol.ConnectTheDots()  # guess bonds from interatomic distances

    return [
        Atoms(numbers=[a.GetAtomicNum() for a in ob.OBMolAtomIter(frag)],
              positions=[(a.GetX(), a.GetY(), a.GetZ()) for a in ob.OBMolAtomIter(frag)])
        for frag in mol.Separate()
    ]

In [5]:
# load the original bimolecular combination 
combination = "combination_23"
loadfn(f"data/{combination}/combination.json")

{'23': ['N2', 'O2']}

In [6]:
# xtb trj file which is an xyz file 
# the first index is the initial point which should match the above cell
atoms = read(f"data/{combination}/0.xtb.trj",format="xyz",index=0) 

In [7]:
seperate_atoms(atoms)

[Atoms(symbols='N2', pbc=False),
 Atoms(symbols='N2', pbc=False),
 Atoms(symbols='N2', pbc=False),
 Atoms(symbols='N2', pbc=False),
 Atoms(symbols='O2', pbc=False),
 Atoms(symbols='O2', pbc=False),
 Atoms(symbols='O2', pbc=False),
 Atoms(symbols='O2', pbc=False)]

### 2. generate some SMILES from the fragments 

In [8]:
from ase import Atoms
from openbabel import openbabel as ob

def to_obmol(atoms: Atoms) -> ob.OBMol:
    mol = ob.OBMol()
    for Z, pos in zip(atoms.numbers, atoms.positions):
        a = mol.NewAtom()
        a.SetAtomicNum(int(Z))
        a.SetVector(*map(float, pos))
    mol.ConnectTheDots()
    mol.PerceiveBondOrders()  # needed for correct double/aromatic bonds
    return mol

def fragment_smiles(atoms: Atoms, fmt: str = "can") -> list[str]:
    """Canonical SMILES for each disconnected fragment."""
    conv = ob.OBConversion()
    conv.SetOutFormat(fmt)
    return [conv.WriteString(frag).split()[0] for frag in to_obmol(atoms).Separate()]

In [9]:
fragments = seperate_atoms(atoms)
fragment_smiles(fragments[0]),fragment_smiles(fragments[1])

(['[N][N]'], ['[N][N]'])

### 3. Read in all frames of the XTB MD run and find new compounds 

In [ ]:
frames =read(f"data/{combination}/0.xtb.trj",format="xyz",index=":")
len(frames)

In [ ]:
# read in the metadat which tells you the "reactants"

metadata = loadfn(f"data/{combination}/combination.json")
reactants = list(metadata.values())[0]

reactants

In [ ]:
# run seperate_atoms for all frames and document unique structures and their appearance 
unique_structures = defaultdict(list)

for i,frame in tqdm(enumerate(frames),total=len(frames)):
    atoms_list = seperate_atoms(frame)
    for at in atoms_list:
        unique_structures["compositions"].append(at.get_chemical_formula()) 
        unique_structures["atoms"].append(at) 
        unique_structures["smiles"].append(fragment_smiles(at)[0])

Counter(unique_structures["compositions"])

In [ ]:
Counter(unique_structures["smiles"])
# as each is an unrelaxed snapshot, we only have 

In [ ]:
df = pd.DataFrame(unique_structures)
df

### 4 relax all fragmented structures 

In [ ]:
#from tblite.ase import TBLite

## uses the same as XTB 
#def make_calc(frag):
#    mult = 2 if sum(frag.numbers) % 2 else 1
#    return TBLite(method="GFN2-xTB", multiplicity=mult, verbosity=0)
from ase.optimize import BFGS
from aimnet.calculators import AIMNet2ASE

def relax_atoms(calculator, atoms: Atoms, fmax=0.05, steps=200):
    atoms.calc = calculator
    BFGS(atoms, logfile=None).run(fmax=fmax, steps=200)
    return atoms


calculator = AIMNet2ASE("aimnet2")

In [ ]:
relaxed_dict = defaultdict(list)
for frame in tqdm(df.T,total=len(df)):
    atoms = df.T[frame]["atoms"]
    atoms.calc = relax_atoms(calculator,atoms)
    relaxed_dict["relaxed_atoms"].append(atoms)
    relaxed_dict["relaxed_smiles"].append(fragment_smiles(atoms)[0])

In [ ]:
df["relaxed_smiles"] = relaxed_dict["relaxed_smiles"]
df["relaxed_atoms"] = relaxed_dict["relaxed_atoms"]

Counter(df["relaxed_smiles"])
#df[df["relaxed_smiles"] == '[O]N1OO1'] 

### 4. Alternatively symmetrise the molecule 

In [ ]:
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.symmetry.analyzer import PointGroupAnalyzer, iterative_symmetrize

def symmetrize(atoms: Atoms, tolerance: float = 0.3) -> tuple[Atoms, str]:
    """Return a symmetrised copy of a molecule and its point group."""
    mol = AseAtomsAdaptor.get_molecule(atoms, charge_spin_check=False)
    sym_mol = iterative_symmetrize(mol, max_n=10, tolerance=tolerance)["sym_mol"]
    pg = PointGroupAnalyzer(sym_mol, tolerance=tolerance).sch_symbol
    return AseAtomsAdaptor.get_atoms(sym_mol),pg 

In [ ]:
sym,pg = symmetrize(fragments[0])
sym,pg

In [ ]:
symmetrised_dict = defaultdict(list)
for frame in tqdm(df.T,total=len(df)):
    atoms = df.T[frame]["atoms"]
    try:
        sym,pg = symmetrize(atoms)
    except:
        pass
    symmetrised_dict["symmetrised_atoms"].append(sym)
    symmetrised_dict["symmetrised_smiles"].append(fragment_smiles(sym)[0])
    symmetrised_dict["pg"].append(pg)

In [ ]:
df["symmetrised_smiles"] = symmetrised_dict["symmetrised_smiles"]
df["symmetrised_atoms"] = symmetrised_dict["symmetrised_atoms"]
df["point_groups"] = symmetrised_dict["pg"]

Counter(df["symmetrised_smiles"])
#df[df["relaxed_smiles"] == '[O]N1OO1'] 

In [ ]:
Counter(df["point_groups"])

## Building a Class for Kinetics 

In [10]:
from arcskin.nanoreactor_analysis import AnalyseNanoReactor
from ase import Atoms 
from ase.io import read 
from openbabel import openbabel as ob 
from monty.serialization import loadfn 
from collections import Counter,defaultdict
from tqdm.auto import tqdm 
import pandas as pd 
import glob 
from ase import Atoms
from openbabel import openbabel as ob

def to_obmol(atoms: Atoms) -> ob.OBMol:
    mol = ob.OBMol()
    for Z, pos in zip(atoms.numbers, atoms.positions):
        a = mol.NewAtom()
        a.SetAtomicNum(int(Z))
        a.SetVector(*map(float, pos))
    mol.ConnectTheDots()
    mol.PerceiveBondOrders()  # needed for correct double/aromatic bonds
    return mol

def fragment_smiles(atoms: Atoms, fmt: str = "can") -> list[str]:
    """Canonical SMILES for each disconnected fragment."""
    conv = ob.OBConversion()
    conv.SetOutFormat(fmt)
    return [conv.WriteString(frag).split()[0] for frag in to_obmol(atoms).Separate()]


In [109]:
combination = "combination_38"
atoms = read(f"data/{combination}/0.xtb.trj",format="xyz",index=0) 
smiles = list(Counter(fragment_smiles(atoms)))
loadfn(f"data/{combination}/combination.json"),smiles

({'38': ['O2', 'SO2']}, ['O=O', 'O=S=O'])

In [110]:
anr = AnalyseNanoReactor(skip_step=100,reactants = smiles)
anr.read_trajectory([f"data/{combination}/{i}.xtb.trj" for i in range(3)],format="xyz")

In [111]:
anr.analyse_trajectories()

 12%|█▎        | 5/40 [00:06<00:53,  1.54s/it][11:53:17] UFFTYPER: Unrecognized atom type: S_5+4 (3)
[11:53:17] UFFTYPER: Unrecognized atom type: S_5+4 (6)
 18%|█▊        | 7/40 [00:10<01:04,  1.96s/it][11:53:21] UFFTYPER: Unrecognized atom type: S_5+4 (1)
[11:53:21] UFFTYPER: Unrecognized atom type: S_5+4 (5)
 20%|██        | 8/40 [00:12<00:58,  1.83s/it][11:53:22] UFFTYPER: Unrecognized atom type: S_5+4 (1)
[11:53:22] UFFTYPER: Unrecognized atom type: S_5+4 (5)
[11:53:22] UFFTYPER: Unrecognized atom type: S_5+4 (1)
[11:53:22] UFFTYPER: Unrecognized atom type: S_5+4 (4)
 22%|██▎       | 9/40 [00:13<00:51,  1.67s/it][11:53:23] UFFTYPER: Unrecognized atom type: S_5+4 (1)
[11:53:23] UFFTYPER: Unrecognized atom type: S_5+4 (5)
[11:53:23] UFFTYPER: Unrecognized atom type: S_5+4 (1)
[11:53:23] UFFTYPER: Unrecognized atom type: S_5+4 (4)
 25%|██▌       | 10/40 [00:14<00:44,  1.47s/it][11:53:24] UFFTYPER: Unrecognized atom type: S_5+4 (1)
[11:53:24] UFFTYPER: Unrecognized atom type: S_5+4 (5)

In [113]:
unique_compositions_counter = pd.DataFrame(anr.unique_smiles_counter).sum(axis=1).sort_values(ascending=False)
unique_compositions_counter

O=S(=O)=O          103.0
[O]O[O]             77.0
[O]S1([O])OO1       24.0
O=S1OO1             18.0
[O]OO[O]             6.0
[O]SO[O]             2.0
[O]O[S@](=O)[O]      1.0
dtype: float64

In [114]:
from collections import Counter 
def combinations_for(available, possible):
    possible = [Counter(p) for p in possible]
    results = []

    def search(remaining, start, chosen):
        if not remaining:                      # everything used up
            results.append(chosen)
            return
        for i in range(start, len(possible)):
            if possible[i] <= remaining:       # enough of each element left
                search(remaining - possible[i], i, chosen + [i])

    search(Counter(available), 0, [])
    return results

# no + so3
available = {"S": 1, "O":4}
possible = [anr.get_num_atoms(smiles) for smiles in sorted(list(unique_compositions_counter.index))]

print(combinations_for(available, possible))

[[4], [5]]


[12:00:01] UFFTYPER: Unrecognized atom type: S_5+4 (1)


In [115]:
print(pd.Series(sorted(unique_compositions_counter.index)).to_string())

0          O=S(=O)=O
1            O=S1OO1
2           [O]OO[O]
3            [O]O[O]
4    [O]O[S@](=O)[O]
5      [O]S1([O])OO1
6           [O]SO[O]


In [117]:
from arcskin.nanoreactor_run import smiles_to_atoms
smiles_to_atoms("[O]S1([O])OO1")[0].write("test.xyz")
!open test.xyz

[12:00:36] UFFTYPER: Unrecognized atom type: S_5+4 (1)
